# Full pipeline run on Kaggle

Runs the complete SFT -> DPO -> comparison pipeline for real headline results, as a Kaggle **background commit** rather than an interactive session — this is the point of using Kaggle instead of Colab: `Save Version -> Save & Run All (Commit)` runs top to bottom on Kaggle's servers, independent of your browser tab, so it survives you closing the laptop.

**Before running, in the notebook Settings panel (right sidebar):**
- **Accelerator**: GPU T4 x2 or P100
- **Internet**: On (required to clone the repo, install packages, and download the base model / dataset)

**Session limits**: 12 hours per session, ~30 GPU-hours/week total. The full pipeline (data prep, SFT, DPO, full-split comparison) previously took about 3h20m end to end on a Colab T4 — well inside a single Kaggle session.

## 1. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Settings (right sidebar).'
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())

## 2. Clone and install

Idempotent: safe to re-run. Unlike Colab, there is no Drive-mount step here — `/kaggle/working` is itself the persistent store: everything left there at the end of the run becomes this notebook version's committed Output automatically.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Hydaspex/qwen3-financial-sft-unsloth-dpo.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/qwen3-financial-sft-unsloth-dpo')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q --branch $BRANCH $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
!pip install -q -e ".[dev]"

# A live kernel's sys.path is only set up from installed .pth/editable-hook
# files at interpreter startup, so a pip install -e run mid-session doesn't
# always make 'finpost' importable without this explicit fallback.
src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from finpost.config import load_config
print('finpost imported successfully from', Path.cwd())

## 3. Configure the full run

Unlike the Colab notebook (which deliberately runs a small smoke config), this uses the committed config as-is — `data.max_samples: null` already means the full TAT-QA split. The only override needed is an absolute MLflow store path inside `/kaggle/working`, so it survives being packaged into the commit's Output rather than resolving relative to a working directory that may differ across cells.

In [ ]:
import yaml

config_path = Path('configs/post_training.yaml')
config = yaml.safe_load(config_path.read_text())

MLFLOW_DB = (REPO_DIR / 'mlflow.db').resolve()
config['mlflow']['tracking_uri'] = f'sqlite:////{MLFLOW_DB}'

config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 4. Prepare TAT-QA data

In [ ]:
!python scripts/prepare_data.py --config configs/post_training.yaml

## 5. Unsloth QLoRA SFT

Checkpoints every 100 steps and auto-resumes if this cell is interrupted and re-run within the *same* session (e.g. an accidental kernel restart) — `outputs/qwen3-financial-sft` persists in `/kaggle/working` for the rest of this session.

This does **not** carry across separate committed runs: each `Save & Run All` starts from a fresh container. If a run is cut off by the 12-hour session cap, the recovery path is to add the previous version's Output as an input dataset to a new notebook version, copy the checkpoint into `outputs/qwen3-financial-sft` before this cell runs, and re-run — not expected to be necessary given the ~3h20m estimate, but worth knowing before starting a long unattended run.

In [ ]:
!python scripts/train_sft.py --config configs/post_training.yaml

## 6. DPO preference optimisation

Same checkpoint/resume behavior as the SFT stage.

In [ ]:
!python scripts/train_dpo.py --config configs/post_training.yaml --sft-adapter outputs/qwen3-financial-sft

## 7. Check artefacts

In [ ]:
for path in [Path('outputs/qwen3-financial-sft'), Path('outputs/qwen3-financial-dpo')]:
    print(path, 'exists:', path.exists())
    if path.exists():
        for item in list(path.iterdir())[:10]:
            print('  ', item.name)

## 8. Compare base, SFT and DPO on the full validation split

This is the headline-numbers step: batched generation, full split (`--limit 0`, the default), bootstrap confidence intervals, and per-example predictions logged to MLflow.

In [ ]:
!python scripts/compare_models.py \
    --config configs/post_training.yaml \
    --sft-adapter outputs/qwen3-financial-sft \
    --dpo-adapter outputs/qwen3-financial-dpo \
    --batch-size 8

## 9. Review runs in MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])

experiments = mlflow.search_experiments()
print(f'Found {len(experiments)} experiments.')

runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in experiments])
wanted = [
    'tags.mlflow.runName', 'metrics.combined', 'metrics.numeric_em', 'metrics.span_match',
    'metrics.numeric_em_ci_low', 'metrics.numeric_em_ci_high',
]

if runs.empty:
    print(f'No runs found at {mlflow.get_tracking_uri()}.')
else:
    present = [c for c in wanted if c in runs.columns]
    if 'metrics.combined' in present:
        table = runs[present].rename(columns={'tags.mlflow.runName': 'run_name'})
        print(table.dropna(subset=['metrics.combined']).sort_values('metrics.combined', ascending=False))
    else:
        print('Runs found but missing metric columns. Columns:', runs.columns.tolist())

## After this run

Record the actual GPU type, wall-clock time and peak VRAM alongside the results — the README's Results section should cite these, not just the numbers, so the experiment is reproducible.

To run this unattended: `Save Version -> Save & Run All (Commit)` from the notebook menu. Progress and final output are visible under the notebook's Output tab once it completes; the MLflow sqlite store (`mlflow.db`) and prediction JSONL files are included in that Output.